In [0]:
# Read bronze parquet datasets

df_transactions = spark.read.parquet(
    "/Volumes/workspace/default/bronze/dbo.transactions.parquet"
)

df_products = spark.read.parquet(
    "/Volumes/workspace/default/bronze/dbo.products.parquet"
)

df_stores = spark.read.parquet(
    "/Volumes/workspace/default/bronze/dbo.stores.parquet"
)

df_customers = spark.read.parquet(
    "/Volumes/workspace/default/bronze/customer.parquet"
)


In [0]:
#display bronze layer 

display(df_transactions)
display(df_products)
display(df_stores)
display(df_customers)


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.00
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.00
6,Water Bottle,Fitness,299.00
7,Smartwatch,Electronics,4999.00
8,Desk Organizer,Accessories,399.00
9,Dumbbell Set,Fitness,1999.00
10,Pen Drive 32GB,Electronics,599.00


store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai


customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


In [0]:
#Silver Layer Cleaning

from pyspark.sql.functions import col

# Clean Transactions
silver_transactions = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

# Clean Products
silver_products = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

# Clean Stores
silver_stores = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

# Clean Customers
silver_customers = df_customers.select(
    col("customer_id").cast("int"),
    "first_name",
    "last_name",
    "email",
    "city",
    col("registration_date").cast("date")
).dropDuplicates(["customer_id"])

In [0]:
#%sql
#CREATE VOLUME workspace.default.silver;
#CREATE VOLUME workspace.default.gold;

In [0]:

#Create Separate Silver Delta Tables

silver_transactions.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_transactions")

silver_products.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_products")

silver_stores.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_stores")

silver_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customers")


# DBTITLE 1,Query Silver Tables

# MAGIC %sql
# MAGIC SELECT * FROM silver_transactions;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM silver_products;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM silver_stores;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM silver_customers;

In [0]:
# DBTITLE 1,Create Gold Layer Dataset

from pyspark.sql.functions import sum, countDistinct, avg

gold_df = silver_transactions \
    .join(silver_customers, "customer_id") \
    .join(silver_products, "product_id") \
    .join(silver_stores, "store_id") \
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )


In [0]:
# DBTITLE 1,Gold Layer Aggregations

gold_summary = gold_df.groupBy(
    "transaction_date",
    "product_id",
    "product_name",
    "category",
    "store_id",
    "store_name",
    "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)

In [0]:
# DBTITLE 1,Display Gold Data

display(gold_summary)

# COMMAND ----------
# DBTITLE 1,Create Gold Delta Table

gold_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("retail_gold_sales_summary")



transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-11-12,3,Yoga Mat,Fitness,4,Downtown Mini Store,Pune,5,2495.0,1,2495.0
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98


In [0]:
#Query Gold Table
%sql
SELECT * FROM retail_gold_sales_summary;

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-11-12,3,Yoga Mat,Fitness,4,Downtown Mini Store,Pune,5,2495.0,1,2495.0
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
